In [11]:
import numpy as np
from PIL import Image
from sklearn.datasets import load_iris

# Практические задания

#### 1. Свой PCA. Реализуйте PCA через np.linalg.eigh от ковариационной матрицы. Сравните результат с PCA через SVD на Iris.

In [19]:
# Создаем функцию для eigh
def pca_via_eigh(X):
    # Центрирование данных
    X_centered = X - X.mean(axis=0)

    # Вычисление ковариационной матрицы
    n = X_centered.shape[0]
    C = (X_centered.T @ X_centered) / (n - 1)

    # Находим собственные значения и векторы
    Lambda, V = np.linalg.eigh(C)

    # Сортировка по убыванию дисперсии
    idx = np.argsort(-Lambda)
    Lambda_sorted = Lambda[idx]
    V_sorted = V[:, idx]

    # Проекция данных
    X_pca = X_centered @ V_sorted

    return X_pca, V_sorted, Lambda_sorted

# Создаем функцию для SVD
def pca_via_svd(X):
    # Центрирование данных
    X_centered = X - X.mean(axis=0)
    n = X_centered.shape[0]
    U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)

    # Проекции: первые k компонент
    X_pca = U * s
    Lambda_svd = (s ** 2) / (n - 1)
    V_svd = Vt.T
    
    return X_pca, V_svd, Lambda_svd

# Основной поток: загрузка датасета (только признаки)
X, _ = load_iris(return_X_y=True)
X_eig, V_eig, Lambda_eig = pca_via_eigh(X)
X_svd, V_svd, Lambda_svd = pca_via_svd(X)

# Сравнение:
for i in range(V_eig.shape[1]):
    dot_product = np.dot(V_eig[:, i], V_svd[:, i])
    # Сравниваем знак первого элемента компоненты
    if dot_product < 0:
        V_eig[:, i] *= - 1
        X_eig[:, i] *= - 1

# Проверка совпадения проекций
assert np.allclose(X_eig, X_svd, atol=1e-10), "Проекции не совпадают!"
# # Проверка собственных значений
assert np.allclose(Lambda_eig, Lambda_svd, atol=1e-10), "Собственные значения не совпадают!"

print("PCA через eigh и SVD совпадают!")
print("Первые 3 собственных значения (eigh):", Lambda_eig[:3])
print("Первые 3 собственных значения (svd):", Lambda_svd[:3])
print("Норма Фробениуса разности проекций:", np.linalg.norm(X_eig - X_svd, "fro"))

PCA через eigh и SVD совпадают!
Первые 3 собственных значения (eigh): [4.22824171 0.24267075 0.0782095 ]
Первые 3 собственных значения (svd): [4.22824171 0.24267075 0.0782095 ]
Норма Фробениуса разности проекций: 2.963802894474141e-14


#### 2. Сжатие изображения. Возьмите фото 1000×1000. Постройте графики ошибки восстановления и размера данных для k = 5, 10, 50, 100, 500. Когда визуально перестаёт быть отличимо от оригинала?

#### 3. Eigenfaces. Загрузите датасет LFW (лица). Сделайте PCA с k=50. Визуализируйте первые 16 компонент как «средние лица».

#### 4. Степенной метод. Реализуйте поиск максимального собственного числа итеративно: v ← Av / ||Av||. Сравните с np.linalg.eig.

#### 5. Рекомендации через SVD. Возьмите матрицу user × movie с пропусками (MovieLens). Сделайте SVD на заполненной нулями версии, выберите k=20. Заполните пропуски как U Σ V^T. Посчитайте MAE на тестовой выборке.

#### 6. LSA на текстах. Постройте TF-IDF матрицу для 100 коротких документов. Сделайте SVD, оставьте 10 компонент. Найдите близкие документы в этом пространстве.

#### 7. Условное число. Посчитайте s[0] / s[-1] для матриц с разной обусловленностью. Покажите, как растёт ошибка при решении плохо обусловленных систем.

#### 8. Truncated SVD vs PCA. Сравните sklearn.TruncatedSVD и sklearn.PCA на разреженной матрице. В чём отличие?